# Step 13: Spatial and Ranking Feedback

**Hypothesis:** A changes neighbour differences and hence B's target priorities, without an immediate density-band handoff.

First observe B's proposed targets immediately before and after A on the same step. Then replace B's spread-score ranking with random ranking among the **same eligible cells**, retaining thresholds, rates, capacity, and budget limiting. Compare AB/BA with the original ranking and three random seeds (8 simulations).

In [1]:
from pathlib import Path
import os
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

current = Path.cwd().resolve()
repo_root = next(p for p in (current, *current.parents) if (p / "src").is_dir())
os.chdir(repo_root)
from src.data_loader import load_locality_data
from src.weed_experiment import ExperimentConfig, run_experiment
from src.experiment_schedules import (
    PolicyParameters, ScheduledPolicy, BudgetLimitedPolicy,
    make_scheduled_policy, standard_schedules, budget_exhaustion_step,
)

SAMPLE_ID = 2036
STEPS = 100  # Change to 260 to inspect a longer horizon.
row = pd.read_csv(repo_root / "results/strong_parrondo_finalists.csv").set_index("sample_id").loc[SAMPLE_ID]
budget = float(row.budget) * STEPS / 100
parameters = PolicyParameters(
    a_rate=float(row.a_rate), b_rate=float(row.b_rate),
    max_cells=int(row.max_cells), boundary=float(row.boundary),
    b_lower=float(row.b_lower), unit_cost=200.0,
)
config = ExperimentConfig(steps=STEPS, growth_rate=float(row.growth_rate), diffusion_rate=float(row.diffusion_rate))
localities = load_locality_data(repo_root / "data")
metro = localities[localities.postcode.astype(int).between(6000, 6199)].copy()
schedules = standard_schedules(STEPS)
print(f"Candidate {SAMPLE_ID}; steps={STEPS}; budget ceiling={budget:,.0f}")


Candidate 2036; steps=100; budget ceiling=250,000


C:\Users\kazut\Documents\ChatGPT\cits4403_computational_model\src\data_loader.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  census["postcode"] = (


In [2]:
def outward_score(grid, mask):
    height, width = grid.shape
    padded = np.pad(grid, 1, constant_values=np.nan)
    valid = np.pad(mask, 1, constant_values=False)
    score = np.zeros_like(grid, dtype=np.float32)
    for dy, dx in ((-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)):
        neighbour = padded[1+dy:1+dy+height, 1+dx:1+dx+width]
        neighbour_valid = valid[1+dy:1+dy+height, 1+dx:1+dx+width]
        difference = np.maximum(grid - neighbour, 0)
        difference[~(mask & neighbour_valid)] = 0
        score += difference
    return score

class RandomRankB:
    def __init__(self, seed):
        self.rng = np.random.default_rng(seed)

    def __call__(self, grid, mask):
        score = outward_score(grid, mask)
        eligible = np.flatnonzero(mask & (grid >= parameters.b_lower) & (grid < parameters.boundary) & (score > 0))
        selected = self.rng.permutation(eligible)[:parameters.max_cells]
        removal = np.zeros_like(grid, dtype=np.float32)
        removal.ravel()[selected] = grid.ravel()[selected] * parameters.b_rate
        return removal, float(removal.sum() * parameters.unit_cost)

class FeedbackObserver:
    def __init__(self, schedule):
        self.schedule = schedule
        self.inner = make_scheduled_policy(schedule, parameters, budget)
        self.probe_b = parameters.make_policy_b()
        self.reset()

    def reset(self):
        self.inner.reset()
        self.index = 0
        self.events = []

    def __call__(self, grid, mask):
        removal, cost = self.inner(grid, mask)
        if self.schedule[self.index] == "A":
            after = grid - removal
            before_targets = self.probe_b(grid, mask)[0] > 0
            after_targets = self.probe_b(after, mask)[0] > 0
            union = int((before_targets | after_targets).sum())
            difference = np.abs(outward_score(after, mask) - outward_score(grid, mask))
            self.events.append({"step": self.index + 1,
                "changed_B_targets": int((before_targets ^ after_targets).sum()),
                "target_jaccard": float((before_targets & after_targets).sum() / union) if union else np.nan,
                "mean_abs_score_change": float(difference[mask].mean())})
        self.index += 1
        return removal, cost

rows, event_tables = [], []
for strategy in ("AB", "BA"):
    schedule = schedules[strategy]
    observer = FeedbackObserver(schedule)
    original = run_experiment(strategy, metro, config, observer)
    event_tables.append(pd.DataFrame(observer.events).assign(strategy=strategy, sample_id=SAMPLE_ID))
    rows.append({"sample_id": SAMPLE_ID, "steps": STEPS, "strategy": strategy, "ranking": "spread score", "seed": None,
                 "relative_change": original.relative_change, "final_cost": original.final_cost})
    for seed in range(3):
        scheduled = ScheduledPolicy(schedule, parameters)
        scheduled.policy_b = RandomRankB(seed)
        result = run_experiment(strategy, metro, config, BudgetLimitedPolicy(scheduled, budget))
        rows.append({"sample_id": SAMPLE_ID, "steps": STEPS, "strategy": strategy, "ranking": "random", "seed": seed,
                     "relative_change": result.relative_change, "final_cost": result.final_cost})
summary = pd.DataFrame(rows)
events = pd.concat(event_tables, ignore_index=True)
display(events.groupby("strategy")[["changed_B_targets", "target_jaccard", "mean_abs_score_change"]].mean())
display(summary.round(6))
summary.to_csv(repo_root / "results/step13_ranking_summary.csv", index=False)
events.to_csv(repo_root / "results/step13_feedback_events.csv", index=False)


,changed_B_targets,target_jaccard,mean_abs_score_change
strategy,,,
AB,0.0,1.0,0.002620
BA,0.0,1.0,0.002593


,sample_id,steps,strategy,ranking,seed,relative_change,final_cost
0,2036,100,AB,spread score,NaN,-0.002064,228212.656250
1,2036,100,AB,random,0.0,0.014860,244391.515625
2,2036,100,AB,random,1.0,0.016063,245474.390625
3,2036,100,AB,random,2.0,0.015679,245014.765625
4,2036,100,BA,spread score,NaN,-0.003486,229166.359375
5,2036,100,BA,random,0.0,0.017112,245957.671875
6,2036,100,BA,random,1.0,0.016669,245404.734375
7,2036,100,BA,random,2.0,0.018355,247928.093750


## Observed results: candidate 2036, 100 steps

**Immediate feedback probe:** A changed outward-spread scores, with mean absolute changes of about **0.002620 (AB)** and **0.002593 (BA)**. However, the selected B target sets before and immediately after A were identical at every observed A step: **zero changed targets; Jaccard overlap 1.0**. Score changes therefore did not translate into a different selected set in this probe. Rankings within that set and delayed effects were not measured.

**Ranking intervention:** Original spread-score ranking improved the final mean by **0.206% (AB)** and **0.349% (BA)**. Random ranking worsened it in all six runs: **+1.486% to +1.606% for AB**, and **+1.667% to +1.835% for BA**.

Random ranking spent about **244,392–247,928**, compared with **228,213–229,166** for the original ranking. Every run remained below the 250,000 ceiling. Random targeting therefore performed worse despite greater total expenditure and removal.

**Interpretation:** Selecting B's highest outward-spread-score cells matters for these mixed schedules in the tested runs. This is stronger evidence for effective B targeting than for the specific claim that A immediately changes B's selected targets: that immediate target-set change was not observed.

The eligibility rule is the same in both versions, but their evolving states and later eligible sets can diverge. The intervention changes targeting, removal amounts, and timing together. Three seeds per schedule are a diagnostic, not a broad statistical conclusion, and this experiment does not isolate the cause of the single-versus-mixed reversal.